### Start a spark session

In [11]:
from pyspark.sql import SparkSession
import os
from pyspark.sql import functions as F
from pyspark.sql import Window


# Spark will download the driver from Maven Central on startup
spark = SparkSession.builder \
    .appName("CBN_Gold") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.2") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hdfs-namenode:9000") \
    .getOrCreate()


# 1. Load your Silver Assets
df_usd_daily = spark.read.parquet("hdfs://hdfs-namenode:9000/cbn_project/silver/forex_daily")
df_cpi_monthly = spark.read.parquet("hdfs://hdfs-namenode:9000/cbn_project/silver/cpi_monthly")

# 2. Aggregate USD to Monthly Grain
# We calculate the Average (trend) and StdDev (stability)
df_usd_monthly = df_usd_daily.withColumn("year", F.year("rate_date")) \
    .withColumn("month", F.month("rate_date")) \
    .groupBy("year", "month") \
    .agg(
        F.round(F.avg("central_rate"), 2).alias("avg_usd_rate"),
        F.round(F.stddev("central_rate"), 2).alias("usd_volatility"),
        F.max("central_rate").alias("peak_usd_rate")
    )

# 3. Join with CPI Data
# We use an inner join to ensure we only have months where both metrics exist
df_merged = df_usd_monthly.join(
    df_cpi_monthly.withColumn("year", F.year("report_date")).withColumn("month", F.month("report_date")),
    on=["year", "month"],
    how="inner"
).select(
    "year", "month", "avg_usd_rate", "usd_volatility", 
    "headline_inflation", "food_inflation"
).orderBy(F.col("year").desc(), F.col("month").desc())


#Define window to look at the earliest data (2006) for comparison, get base value and get growth index
window_start = Window.orderBy("year", "month")

df_final = df_merged.withColumn("first_usd", F.first("avg_usd_rate").over(window_start)) \
    .withColumn("first_cpi", F.first("headline_inflation").over(window_start)) \
    .withColumn("usd_growth_index", F.round((F.col("avg_usd_rate") / F.col("first_usd")) * 100, 2)) \
    .withColumn("inflation_growth_index", F.round((F.col("headline_inflation") / F.col("first_cpi")) * 100, 2)) \
    .select("year", "month", "avg_usd_rate", "headline_inflation", "usd_growth_index", "inflation_growth_index") \
    .orderBy(F.col("year").desc(), F.col("month").desc())

# create a postgres database connection
jdbc_url = "jdbc:postgresql://postgres:5432/cbn_analytics"
db_properties = {
    "user": "fensals",
    "password": "fensals",
    "driver": "org.postgresql.Driver"
}

# write the gold table to postgres
df_final.write.jdbc(
    url=jdbc_url, 
    table="gold_macro_economic_data", 
    mode="overwrite", 
    properties=db_properties
)

print("Gold data successfully migrated to PostgreSQL!")

🚀 Gold data successfully migrated to PostgreSQL Serving Layer!
